<a href="https://colab.research.google.com/github/updalla-apshir/Transfer-Learning-with-Pretrained-Models/blob/main/Transfer_Learning_with_Pretrained_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
import matplotlib.pyplot as plt
import time
import tqdm

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229,0.224,0.225]
        )
    ]
)

In [ ]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)
testing_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 13.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 204kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.79MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 28.6MB/s]


In [ ]:
training_dataloader = DataLoader(
    training_data,
    batch_size=32,
    shuffle=True

)
testing_dataloader = DataLoader(
    testing_data,
    batch_size=32,
    shuffle=False
)

In [ ]:
class_names = training_data.classes
class_names

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [ ]:
import torch.nn as nn

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features,len(class_names))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s]


In [ ]:
model = model.to(device)

In [ ]:
image, label = training_data[120]
image.shape

torch.Size([3, 224, 224])

In [ ]:
class_names[label]

'Sandal'

## Evaluate model accuracy on the full test set

In [ ]:
print("\n--- Experiment: Pretrained baseline (Evaluation only) ---")

start_time_eval_baseline = time.time()
model.eval()

correct = 0
total = 0

with torch.inference_mode():

    for images, labels in testing_dataloader:
        images = images.to(device)
        labels = labels.to(device)


        y_logits = model(images)

        y_pred = y_logits.argmax(dim=1)

        correct += (y_pred == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total
end_time_eval_baseline = time.time()
duration_eval_baseline = end_time_eval_baseline - start_time_eval_baseline

print(f"Testing Accuracy for non-training model: {accuracy * 100:.2f}%")
print(f"Evaluation runtime: {duration_eval_baseline:.2f} seconds")


--- Experiment: Pretrained baseline (Evaluation only) ---
Testing Accuracy for non-training model: 10.54%
Evaluation runtime: 26.59 seconds


## Freeze Pretrained Model Parameters Then Train Then Test

In [ ]:
# freeze all
for param in model.parameters():
  param.requires_grad = False

# unfreeze the final layer
for param in model.fc.parameters():
  param.requires_grad = True

In [ ]:
print(model.fc.weight.requires_grad)

True


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.1)

In [ ]:
print("\n--- Experiment: Feature extraction (FC only) ---")
start_time_train_fc = time.time()
epochs = 10

for epoch in tqdm.tqdm(range(epochs)):
    train_acc = 0
    train_loss = 0

    model.train()
    for x, y in training_dataloader:

        x, y = x.to(device), y.to(device)

        y_logits = model(x)
        y_pred = y_logits.argmax(dim=1)

        loss = loss_fn(y_logits, y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_acc += (y_pred == y).sum().item()

    train_loss /= len(training_dataloader)
    train_acc /= len(training_data)

    print(f"Epoch {epoch+1}: Loss {train_loss:.4f}, Train Accuracy {train_acc*100:.2f}")
end_time_train_fc = time.time()
duration_train_fc = end_time_train_fc - start_time_train_fc
print(f"Training runtime: {duration_train_fc:.2f} seconds")


--- Experiment: Feature extraction (FC only) ---


 10%|█         | 1/10 [02:29<22:21, 149.06s/it]

Epoch 1: Loss 0.9656, Train Accuracy 76.20


 20%|██        | 2/10 [04:57<19:48, 148.62s/it]

Epoch 2: Loss 0.7271, Train Accuracy 80.26


...

Epoch 8: Loss 0.6669, Train Accuracy 82.29


 90%|█████████ | 9/10 [22:15<02:28, 148.62s/it]

Epoch 9: Loss 0.6672, Train Accuracy 82.06


100%|██████████| 10/10 [24:43<00:00, 148.35s/it]

Epoch 10: Loss 0.6480, Train Accuracy 82.48
Training runtime: 1483.54 seconds


In [ ]:
  # torch.save(model.state_dict(),'model.pth')

In [ ]:
# model = resnet18(weights=ResNet18_Weights.DEFAULT)
# model.fc = nn.Linear(model.fc.in_features,len(class_names))

In [ ]:
# model.load_state_dict(torch.load('model.pth'))

In [ ]:
start_time_eval_fc = time.time()
model.eval()

correct = 0
total = 0

with torch.inference_mode():

    for images, labels in testing_dataloader:
        images = images.to(device)
        labels = labels.to(device)


        y_logits = model(images)

        y_pred = y_logits.argmax(dim=1)

        correct += (y_pred == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total
end_time_eval_fc = time.time()
duration_eval_fc = end_time_eval_fc - start_time_eval_fc

print(f"Testing Accuracy after unfreezing: {accuracy * 100:.2f}%")
print(f"Evaluation runtime: {duration_eval_fc:.2f} seconds")

Testing Accuracy after unfreezing: 80.98%
Evaluation runtime: 24.06 seconds


## Experiment B — Fine-tuning some layers


In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features,len(class_names))
model = model.to(device)

In [ ]:
# unfreeze Some parameters

for param in model.parameters():
  param.requires_grad = False

for param in model.layer4.parameters():
  param.requires_grad = True

for param in model.fc.parameters():
  param.requires_grad = True


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.1)

In [16]:
print("\n--- Experiment: Partial fine-tuning (Layer4 + FC) ---")
start_time_train_partial = time.time()
epochs = 10

for epoch in tqdm.tqdm(range(epochs)):
    train_acc = 0
    train_loss = 0

    model.train()
    for x, y in training_dataloader:

        x, y = x.to(device), y.to(device)

        y_logits = model(x)
        y_pred = y_logits.argmax(dim=1)

        loss = loss_fn(y_logits, y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_acc += (y_pred == y).sum().item()

    train_loss /= len(training_dataloader)
    train_acc /= len(training_data)

    print(f"Epoch {epoch+1}: Loss {train_loss:.4f}, Train Accuracy {train_acc*100:.2f}")
end_time_train_partial = time.time()
duration_train_partial = end_time_train_partial - start_time_train_partial
print(f"Training runtime: {duration_train_partial:.2f} seconds")


--- Experiment: Partial fine-tuning (Layer4 + FC) ---


 10%|█         | 1/10 [02:43<24:34, 163.82s/it]

Epoch 1: Loss 0.3939, Train Accuracy 87.61


 20%|██        | 2/10 [05:25<21:41, 162.63s/it]

Epoch 2: Loss 0.1901, Train Accuracy 93.10


...

Epoch 8: Loss 0.0333, Train Accuracy 98.78


 90%|█████████ | 9/10 [24:23<02:41, 161.99s/it]

Epoch 9: Loss 0.0258, Train Accuracy 99.09


100%|██████████| 10/10 [27:05<00:00, 162.51s/it]

Epoch 10: Loss 0.0218, Train Accuracy 99.21
Training runtime: 1625.06 seconds


In [17]:
start_time_eval_partial = time.time()
model.eval()

correct = 0
total = 0

with torch.inference_mode():

    for images, labels in testing_dataloader:
        images = images.to(device)
        labels = labels.to(device)


        y_logits = model(images)

        y_pred = y_logits.argmax(dim=1)

        correct += (y_pred == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total
end_time_eval_partial = time.time()
duration_eval_partial = end_time_eval_partial - start_time_eval_partial

print(f"Test Accuracy after unfreezing some parameters: {accuracy * 100:.2f}%")
print(f"Evaluation runtime: {duration_eval_partial:.2f} seconds")

Test Accuracy after unfreezing some parameters: 93.02%
Evaluation runtime: 24.02 seconds


## Experiment C — Fine-tuning All layers


In [18]:
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features,len(class_names))
model = model.to(device)

In [20]:
for param in model.parameters():
  param.requires_grad = True

In [21]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.1)

In [22]:
print("\n--- Experiment: Full fine-tuning (All layers) ---")
start_time_train_full = time.time()
epochs = 10

for epoch in tqdm.tqdm(range(epochs)):
    train_acc = 0
    train_loss = 0

    model.train()
    for x, y in training_dataloader:

        x, y = x.to(device), y.to(device)

        y_logits = model(x)
        y_pred = y_logits.argmax(dim=1)

        loss = loss_fn(y_logits, y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_acc += (y_pred == y).sum().item()

    train_loss /= len(training_dataloader)
    train_acc /= len(training_data)

    print(f"Epoch {epoch+1}: Loss {train_loss:.4f}, Train Accuracy {train_acc*100:.2f}")
end_time_train_full = time.time()
duration_train_full = end_time_train_full - start_time_train_full
print(f"Training runtime: {duration_train_full:.2f} seconds")


--- Experiment: Full fine-tuning (All layers) ---


 10%|█         | 1/10 [04:22<39:24, 262.72s/it]

Epoch 1: Loss 0.3981, Train Accuracy 87.21


 20%|██        | 2/10 [08:43<34:51, 261.48s/it]

Epoch 2: Loss 0.2023, Train Accuracy 92.72


...

Epoch 8: Loss 0.0501, Train Accuracy 98.19


 90%|█████████ | 9/10 [39:13<04:22, 262.10s/it]

Epoch 9: Loss 0.0400, Train Accuracy 98.57


100%|██████████| 10/10 [43:35<00:00, 261.51s/it]

Epoch 10: Loss 0.0319, Train Accuracy 98.81
Training runtime: 2615.08 seconds


In [23]:
start_time_eval_full = time.time()
model.eval()

correct = 0
total = 0

with torch.inference_mode():

    for images, labels in testing_dataloader:
        images = images.to(device)
        labels = labels.to(device)


        y_logits = model(images)

        y_pred = y_logits.argmax(dim=1)

        correct += (y_pred == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total
end_time_eval_full = time.time()
duration_eval_full = end_time_eval_full - start_time_eval_full

print(f"Test Accuracy after fully fine tunned: {accuracy * 100:.2f}%")
print(f"Evaluation runtime: {duration_eval_full:.2f} seconds")

Test Accuracy after fully fine tunned: 94.10%
Evaluation runtime: 24.08 seconds


### Experiment Runtimes Summary

In [25]:
print("\n--- Runtimes Summary ---")
print(f"Pretrained baseline (Evaluation only) runtime: {duration_eval_baseline:.2f} seconds")
print(f"Feature extraction (FC only) - Training runtime: {duration_train_fc:.2f} seconds")
print(f"Feature extraction (FC only) - Evaluation runtime: {duration_eval_fc:.2f} seconds")
print(f"Partial fine-tuning (Layer4 + FC) - Training runtime: {duration_train_partial:.2f} seconds")
print(f"Partial fine-tuning (Layer4 + FC) - Evaluation runtime: {duration_eval_partial:.2f} seconds")
print(f"Full fine-tuning (All layers) - Training runtime: {duration_train_full:.2f} seconds")
print(f"Full fine-tuning (All layers) - Evaluation runtime: {duration_eval_full:.2f} seconds")


--- Runtimes Summary ---
Pretrained baseline (Evaluation only) runtime: 26.59 seconds
Feature extraction (FC only) - Training runtime: 1483.54 seconds
Feature extraction (FC only) - Evaluation runtime: 24.06 seconds
Partial fine-tuning (Layer4 + FC) - Training runtime: 1625.06 seconds
Partial fine-tuning (Layer4 + FC) - Evaluation runtime: 24.02 seconds
Full fine-tuning (All layers) - Training runtime: 2615.08 seconds
Full fine-tuning (All layers) - Evaluation runtime: 24.08 seconds


## Transfer Learning Experiment

**Objective**
Investigation into the performance impact of progressive layer unfreezing when adapting a ResNet18 model pretrained on ImageNet to the FashionMNIST classification task.

**Setup**
* **Model:** ResNet18 (weights: `ResNet18_Weights.DEFAULT`)
* **Dataset:** FashionMNIST (resized to 224x224, normalized to ImageNet statistics)
* **Optimizer/Hyperparameters:** SGD, $\eta = 0.1$, 10 epochs
* **Metric:** Top-1 Test Accuracy

**Experiments & Results**

| Configuration       | Trainable Layers | Test Accuracy | Training Runtime (s) | Evaluation Runtime (s) |
| ------------------- | ---------------- | ------------: | -------------------- | ---------------------- |
| Pretrained baseline | None             |        10.54% | None                  | 26.59                  |
| Feature extraction  | FC               |        80.98% | 1483.54              | 24.06                  |
| Partial fine-tuning | Layer4 + FC      |        93.02% | 1625.06              | 24.02                  |
| Full fine-tuning    | All              |        94.10% | 2615.08              | 24.08                  |

**Key Observations**
* **Representation Transfer:** The transition from the untrained baseline to head-only feature extraction resulted in a **+70.44 percentage-point (pp)** accuracy gain (80.98% - 10.54%). This confirms the strong generalizability of ImageNet-trained spatial hierarchies to grayscale fashion iconography.
* **Feature Adaptation:** Unfreezing `layer4` provided a significant **+12.04 pp** improvement over the fixed feature extractor (93.02% vs 80.98%). This indicates that adapting high-level semantic filters to the specific domain manifolds of FashionMNIST is critical for high-precision results.
* **Full Fine-tuning Impact:** Full fine-tuning yielded a further **+1.08 pp** improvement over partial fine-tuning (94.10% vs 93.02%), demonstrating that adapting lower-level features can still contribute to better performance for this task.

**Conclusion**
Full fine-tuning (All layers) achieved the highest accuracy at 94.10%. Progressive unfreezing consistently improved performance, indicating that fine-tuning deeper layers, even with a relatively small dataset like FashionMNIST, can further adapt the model for improved classification accuracy.